In [0]:
#==============DEBUG==============

# Droppare tabella vecchia se esiste
# spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{GOLD}.loan_risk_summary")

In [0]:
from pyspark.sql.functions import col, count, avg, min, max

CATALOG = "czech_fintech"
SILVER = "silver"
GOLD = "gold"

# ============================================================================
# LOAN_RISK_SUMMARY
# Per ogni prestito: id, account, importo, durata, rata, stato, profilo cliente
# ============================================================================

loan = spark.table(f"{CATALOG}.{SILVER}.loan")
account = spark.table(f"{CATALOG}.{SILVER}.account")
client = spark.table(f"{CATALOG}.{SILVER}.client")
district = spark.table(f"{CATALOG}.{SILVER}.district")

loan_risk = (
    loan
    .join(account.select("account_id", "district_id"), "account_id", "left")
    .join(
        district.select(col("A1").alias("district_id_key"), col("A2").alias("district_name")),
        col("district_id") == col("district_id_key"),
        "left"
    ).drop("district_id_key")
)

loan_risk_summary = loan_risk.select(
    col("loan_id"),
    col("account_id"),
    col("amount").cast("decimal(15,2)").alias("loan_amount"),
    col("duration").cast("int").alias("loan_duration_months"),
    (col("amount") / col("duration")).cast("decimal(15,2)").alias("monthly_payment"),
    col("status").alias("loan_status"),
    col("district_id"),
    col("district_name")
)

loan_risk_summary.write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{GOLD}.loan_risk_summary"
)

count = spark.table(f"{CATALOG}.{GOLD}.loan_risk_summary").count()
print(f"✅ loan_risk_summary: {count} rows")
spark.table(f"{CATALOG}.{GOLD}.loan_risk_summary").limit(5).show()